# Bài 3.5 · Optimizer trên dữ liệu thật

**Câu hỏi của buổi này:** cùng một gradient, các optimizer bước đi khác nhau thế nào, và trên một bài thật thì chọn optimizer, learning rate, lịch learning rate ra sao?

| Mục | Sẽ thấy |
|---|---|
| 1. Quỹ đạo trên một mặt loss | SGD, momentum, Adam đi trên thung lũng hẹp |
| 2. Một bước Adam tính tay | khớp với `torch.optim.Adam` |
| 3. So optimizer trên Fashion-MNIST | đường loss và accuracy, cùng số bước |
| 4. Quét learning rate | loss sau 1 epoch theo learning rate của SGD và Adam |
| 5. Scheduler | learning rate và loss khi giữ nguyên, cosine, one-cycle |
| 6. Adam + L2 hay AdamW | độ lớn trọng số và accuracy |
| 7. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 4 phút. Bài giảng: [ai-canuc80k · bài 3.5](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/3.5).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, math
import torch, torchvision
from torch import nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
EPOCHS, N_TRAIN = 4, 60000
table(pd.DataFrame([('torch', torch.__version__), ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Quỹ đạo trên một mặt loss

Mặt loss $L(w_1, w_2) = \tfrac12(w_1^2 + 100\,w_2^2)$ là một thung lũng hẹp: dốc theo $w_2$ gấp 100 lần theo $w_1$. Bắt đầu từ cùng một điểm, chạy 80 bước với ba optimizer, learning rate chọn sao cho SGD vừa chưa nổ.

In [ ]:
def loss2(w): return 0.5 * (w[0] ** 2 + 100 * w[1] ** 2)
def run2(make, steps=80):
    w = torch.tensor([-9.0, 1.6], requires_grad=True); opt = make([w]); path = [w.detach().clone()]
    for _ in range(steps):
        opt.zero_grad(); loss2(w).backward(); opt.step(); path.append(w.detach().clone())
    return torch.stack(path).numpy()

paths = {'SGD, lr 0.019': run2(lambda p: torch.optim.SGD(p, lr=0.019)),
         'SGD + momentum 0.9, lr 0.008': run2(lambda p: torch.optim.SGD(p, lr=0.008, momentum=0.9)),
         'Adam, lr 0.3': run2(lambda p: torch.optim.Adam(p, lr=0.3))}
a, b = np.meshgrid(np.linspace(-10, 3, 200), np.linspace(-2.2, 2.2, 200))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={'width_ratios': [1.6, 1]})
axes[0].contour(a, b, 0.5 * (a ** 2 + 100 * b ** 2), levels=np.geomspace(0.1, 300, 14), colors='#bbb', linewidths=.8)
for (n, P), c in zip(paths.items(), [C['clay'], C['blue'], C['green']]):
    axes[0].plot(P[:, 0], P[:, 1], marker='o', ms=3, lw=1.3, color=c, label=n)
    axes[1].plot([loss2(torch.tensor(p)).item() for p in P], color=c, lw=2, label=n)
axes[0].scatter([0], [0], marker='*', s=200, color='black', zorder=5); axes[0].set(xlabel='$w_1$', ylabel='$w_2$', title='Quỹ đạo 80 bước (sao: điểm tối ưu)'); axes[0].legend(fontsize=9)
axes[1].set_yscale('log'); axes[1].set(xlabel='bước', ylabel='loss (thang log)', title='Loss theo bước')
plt.tight_layout(); plt.show()
final = {n: loss2(torch.tensor(P[-1])).item() for n, P in paths.items()}
note('SGD dội qua dội lại theo hướng dốc w₂ mà tiến chậm theo w₁. Momentum cộng dồn hướng đi nên băng nhanh dọc thung lũng; Adam chia bước theo độ lớn gradient từng tọa độ nên đi gần như thẳng. '
     'Loss sau 80 bước: ' + ', '.join(f'{n.split(",")[0]} <b>{v:.1e}</b>' for n, v in final.items()) + '.')

## 2. Một bước Adam tính tay

$$m \leftarrow \beta_1 m + (1-\beta_1) g,\quad v \leftarrow \beta_2 v + (1-\beta_2) g^2,\quad \hat m = \frac{m}{1-\beta_1^t},\ \hat v = \frac{v}{1-\beta_2^t},\quad w \leftarrow w - \eta\, \frac{\hat m}{\sqrt{\hat v} + \epsilon}$$

Tự tính ba bước trên mặt loss ở mục 1, so với `torch.optim.Adam`.

In [ ]:
lr, b1, b2, eps = 0.5, 0.9, 0.999, 1e-8
w = np.array([-9.0, 1.6]); m = np.zeros(2); v = np.zeros(2); rows = []
wt = torch.tensor([-9.0, 1.6], requires_grad=True); opt = torch.optim.Adam([wt], lr=lr, betas=(b1, b2), eps=eps)
for t in range(1, 4):
    g = np.array([w[0], 100 * w[1]])
    m = b1 * m + (1 - b1) * g; v = b2 * v + (1 - b2) * g ** 2
    w = w - lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps)
    opt.zero_grad(); loss2(wt).backward(); opt.step()
    rows.append((t, *g, *w, *wt.detach().numpy()))
step = pd.DataFrame(rows, columns=['bước', 'g₁', 'g₂', 'w₁ tự tính', 'w₂ tự tính', 'w₁ torch', 'w₂ torch'])
table(step, 'Ba bước Adam, learning rate 0.5', fmt={c: '{:.4f}' for c in step.columns[1:]})
check('tự tính khớp torch.optim.Adam', np.allclose(step[['w₁ tự tính', 'w₂ tự tính']], step[['w₁ torch', 'w₂ torch']], atol=1e-5))
note(f'Bước đầu, m̂ / √v̂ = ±1 ở mọi tọa độ: Adam đi đúng một bước dài <b>0.5</b> theo cả hai chiều, dù gradient theo w₂ ({step.iloc[0, 2]:.0f}) lớn gấp {abs(step.iloc[0, 2] / step.iloc[0, 1]):.1f} lần theo w₁.')

## 3. So optimizer trên Fashion-MNIST

Một CNN nhỏ, cùng khởi tạo, cùng thứ tự batch, cùng 4 epoch. Mỗi optimizer dùng learning rate quen dùng cho nó.

In [ ]:
with quiet():
    tr = torchvision.datasets.FashionMNIST('/tmp/fmnist', train=True, download=True)
    te = torchvision.datasets.FashionMNIST('/tmp/fmnist', train=False, download=True)
X = (tr.data[:N_TRAIN].float() / 255 - 0.286).div(0.353).unsqueeze(1).to(device); Y = tr.targets[:N_TRAIN].to(device)
Xt = (te.data.float() / 255 - 0.286).div(0.353).unsqueeze(1).to(device); Yt = te.targets.to(device)

def make_net():
    torch.manual_seed(0)
    return nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Flatten(), nn.Linear(64 * 49, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)

def test_acc(net):
    net.eval()
    with torch.no_grad():
        return np.mean([(net(Xt[i:i + 2000]).argmax(1) == Yt[i:i + 2000]).float().mean().item() for i in range(0, len(Yt), 2000)])

def train(make_opt, epochs=EPOCHS, sched=None, bs=128):
    net = make_net(); opt = make_opt(net.parameters())
    steps = epochs * math.ceil(len(Y) / bs); sch = sched(opt, steps) if sched else None
    g = torch.Generator().manual_seed(0); losses, lrs = [], []
    for ep in range(epochs):
        net.train()
        for idx in torch.randperm(len(Y), generator=g).to(device).split(bs):
            loss = F.cross_entropy(net(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item()); lrs.append(opt.param_groups[0]['lr'])
            if sch: sch.step()
    return net, np.array(losses), np.array(lrs)

opts = {'SGD, lr 0.05': lambda p: torch.optim.SGD(p, lr=0.05),
        'SGD + momentum, lr 0.05': lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
        'Adam, lr 1e-3': lambda p: torch.optim.Adam(p, lr=1e-3),
        'AdamW, lr 1e-3': lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=0.05)}
res, curves = [], {}
for n, mk in opts.items():
    t = time.time(); net, L, _ = train(mk); curves[n] = L
    res.append((n, L[-200:].mean(), test_acc(net), time.time() - t))
cmp = pd.DataFrame(res, columns=['optimizer', 'loss train (200 bước cuối)', 'test acc', 'giây'])
table(cmp, f'{EPOCHS} epoch, {len(curves[n])} bước mỗi optimizer', fmt={'loss train (200 bước cuối)': '{:.3f}', 'test acc': '{:.4f}', 'giây': '{:.0f}'}, best='test acc')
fig, ax = plt.subplots(figsize=(10, 3.8))
for (n, L), c in zip(curves.items(), [C['gray'], C['clay'], C['blue'], C['green']]):
    ax.plot(pd.Series(L).rolling(50, min_periods=1).mean(), color=c, lw=2, label=n)
ax.set_yscale('log'); ax.set(xlabel='bước', ylabel='loss train (trung bình trượt 50, thang log)', title='Cùng model, cùng dữ liệu, cùng số bước'); ax.legend()
plt.tight_layout(); plt.show()
best = cmp.loc[cmp['test acc'].idxmax()]
note(f"SGD không momentum chậm nhất (loss cuối {cmp.iloc[0, 1]:.3f}); thêm momentum xuống {cmp.iloc[1, 1]:.3f}. Test acc cao nhất: <b>{best['optimizer']}</b> ({best['test acc']:.4f}). "
     f"Khoảng cách giữa các optimizer tốt thường nhỏ hơn khoảng cách do chọn learning rate (mục 4).")

## 4. Quét learning rate

Train 1 epoch với nhiều learning rate, ghi loss train trung bình của 100 bước cuối. Learning rate quá nhỏ thì học chậm; quá lớn thì loss nổ.

In [ ]:
grid = {'SGD + momentum': (lambda lr: (lambda p: torch.optim.SGD(p, lr=lr, momentum=0.9)), np.geomspace(1e-4, 1, 9)),
        'Adam': (lambda lr: (lambda p: torch.optim.Adam(p, lr=lr)), np.geomspace(1e-5, 1e-1, 9))}
sweep = []
for n, (mk, lrs_) in grid.items():
    for lr in lrs_:
        _, L, _ = train(mk(lr), epochs=1)
        sweep.append((n, lr, np.nanmean(L[-100:]) if np.isfinite(L[-100:]).all() else np.nan))
sw = pd.DataFrame(sweep, columns=['optimizer', 'learning rate', 'loss cuối'])
fig, ax = plt.subplots(figsize=(9, 3.8))
for (n, g), c in zip(sw.groupby('optimizer', sort=False), [C['clay'], C['blue']]):
    ax.plot(g['learning rate'], g['loss cuối'].clip(upper=3), marker='o', color=c, lw=2, label=n)
    bl = g.loc[g['loss cuối'].idxmin()]; ax.scatter([bl['learning rate']], [bl['loss cuối']], s=120, facecolors='none', edgecolors=c, lw=2)
ax.axhline(math.log(10), color='#999', ls=':'); ax.text(1.2e-5, math.log(10) + 0.05, 'ln 10: chưa học được gì', fontsize=9, color='#777')
ax.set_xscale('log'); ax.set(xlabel='learning rate (thang log)', ylabel='loss sau 1 epoch', title='Mỗi optimizer có một dải learning rate dùng được'); ax.legend()
plt.tight_layout(); plt.show()
best_lr = sw.loc[sw.groupby('optimizer', sort=False)['loss cuối'].idxmin()]
table(best_lr, 'Learning rate tốt nhất sau 1 epoch', fmt={'learning rate': '{:.0e}', 'loss cuối': '{:.3f}'})
note('Hai dải tốt nằm cách nhau vài bậc lũy thừa 10: learning rate tốt của SGD gần như chắc chắn làm Adam nổ, và ngược lại. Đổi optimizer thì phải quét lại learning rate.')

## 5. Scheduler

Cùng Adam, cùng learning rate đỉnh 3e-3, ba lịch: **giữ nguyên**, **cosine** (giảm dần về 0), **one-cycle** (tăng trong 30% đầu rồi giảm).

In [ ]:
LR = 3e-3
scheds = {'giữ nguyên': None,
          'cosine': lambda o, s: torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=s),
          'one-cycle': lambda o, s: torch.optim.lr_scheduler.OneCycleLR(o, max_lr=LR, total_steps=s)}
srows, sc = [], {}
for n, sch in scheds.items():
    net, L, lrs_ = train(lambda p: torch.optim.Adam(p, lr=LR), sched=sch); sc[n] = (L, lrs_)
    srows.append((n, L[-200:].mean(), test_acc(net)))
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
for (n, (L, lrs_)), c in zip(sc.items(), [C['gray'], C['blue'], C['clay']]):
    axes[0].plot(lrs_, color=c, lw=2, label=n); axes[1].plot(pd.Series(L).rolling(100, min_periods=1).mean(), color=c, lw=2, label=n)
axes[0].set(xlabel='bước', ylabel='learning rate', title='Learning rate theo bước'); axes[1].set(xlabel='bước', ylabel='loss train (trung bình trượt 100)', title='Loss theo bước'); axes[1].set_yscale('log')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()
st = pd.DataFrame(srows, columns=['lịch', 'loss train cuối', 'test acc'])
table(st, f'Adam, learning rate đỉnh {LR}', fmt={'loss train cuối': '{:.3f}', 'test acc': '{:.4f}'}, best='test acc')
bs_ = st.loc[st['loss train cuối'].idxmin()]
note(f"Loss train cuối: giữ nguyên {st.iloc[0, 1]:.3f}, cosine {st.iloc[1, 1]:.3f}, one-cycle {st.iloc[2, 1]:.3f}; thấp nhất là <b>{bs_['lịch']}</b>. "
     + ("Giảm learning rate về cuối giúp model dừng ở đáy thay vì nhảy quanh nó." if bs_['lịch'] != 'giữ nguyên' else "Với ít bước như vậy, lịch giảm dần chưa kịp phát huy: thử tăng số epoch."))

## 6. Adam + L2 hay AdamW

Thêm $\lambda\|w\|^2$ vào loss rồi dùng Adam thì phần phạt cũng bị chia cho $\sqrt{\hat v}$: tham số có gradient lớn gần như không bị phạt. **AdamW** tách riêng: mỗi bước trừ thẳng $\eta\lambda w$, nên hệ số của nó được nhân với learning rate (0.1 × 1e-3). So độ lớn trọng số và accuracy.

In [ ]:
wd = 1e-3
nets = {}
nets['Adam, không phạt'] = train(lambda p: torch.optim.Adam(p, lr=1e-3))[0]
nets['Adam + L2 (weight_decay trong Adam)'] = train(lambda p: torch.optim.Adam(p, lr=1e-3, weight_decay=wd))[0]
nets['AdamW (tách riêng)'] = train(lambda p: torch.optim.AdamW(p, lr=1e-3, weight_decay=wd * 100))[0]
wrow = [(n, torch.sqrt(sum((p ** 2).sum() for p in net.parameters())).item(), test_acc(net)) for n, net in nets.items()]
wdf = pd.DataFrame(wrow, columns=['cách', 'độ lớn trọng số ‖w‖', 'test acc'])
table(wdf, 'Phạt trọng số (AdamW dùng hệ số 0.1 vì nó nhân với learning rate)', fmt={'độ lớn trọng số ‖w‖': '{:.1f}', 'test acc': '{:.4f}'}, best='test acc')
note(f"Không phạt: ‖w‖ = {wdf.iloc[0, 1]:.1f}; Adam + L2: {wdf.iloc[1, 1]:.1f}; AdamW: <b>{wdf.iloc[2, 1]:.1f}</b>. "
     f"Adam + L2 chia cả phần phạt cho √v̂: trọng số ít được dùng (gradient nhỏ) bị kéo về 0 rất mạnh, trọng số được dùng nhiều thì gần như không. AdamW kéo mọi trọng số theo cùng một tỉ lệ.")

## 7. Tóm tắt

In [ ]:
summary = [('mặt loss hẹp, loss sau 80 bước: SGD / momentum / Adam', ' / '.join(f'{v:.1e}' for v in final.values())),
           ('Fashion-MNIST, test acc tốt nhất', f"{best['optimizer']} {best['test acc']:.4f}"),
           ('learning rate tốt nhất: SGD + momentum / Adam', ' / '.join(f'{v:.0e}' for v in best_lr['learning rate'])),
           ('loss train cuối: giữ nguyên / cosine / one-cycle', ' / '.join(f'{v:.3f}' for v in st['loss train cuối'])),
           ('‖w‖: không phạt / Adam + L2 / AdamW', ' / '.join(f'{v:.1f}' for v in wdf.iloc[:, 1]))]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** ở mục 1 tăng learning rate SGD lên 0.085 để thấy nổ; ở mục 3 thêm `torch.optim.RMSprop`; ở mục 5 thêm warmup tuyến tính 5% đầu cho lịch cosine.